In [1]:
"""
crop_half_domain.py
-------------------
Loops over all case folders under the infrared data root directory,
reads window_1_temperature.csv and window_2_temperature.csv from each
  <case>/time_average/average_temperature/
directory, crops to the upper half (top 50% of rows), saves the cropped
CSVs and temperature-map PNG plots to a new subfolder:
  <case>/time_average/average_temperature/half_domain_temperature/
"""

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── USER SETTINGS ──────────────────────────────────────────────────────────────
ROOT_DIR = r"D:\FCAI\plain_coupon\infared_data"
CSV_FILES = ["window_1_temperature.csv", "window_2_temperature.csv"]
OUTPUT_SUBDIR = "half_domain_temperature"
COLORMAP = "inferno"          # good for temperature data; change if preferred
# ───────────────────────────────────────────────────────────────────────────────


def parse_case_name(folder_name: str) -> str:
    """Return a human-readable label from a folder name like cr_625_phi_09."""
    parts = folder_name.split("_")
    try:
        cr  = parts[1]
        phi = parts[3]
        # Insert decimal point: 09 → 0.9, 118 → 1.18, 137 → 1.37
        phi_val = int(phi) / 100 if len(phi) == 3 else int(phi) / 10
        return f"CR = {cr} slpm,  φ = {phi_val:.2f}"
    except (IndexError, ValueError):
        return folder_name


def crop_and_save(csv_path: str, out_dir: str, case_label: str) -> None:
    """Read a temperature CSV, crop upper half, save CSV + PNG."""
    # ── load ──────────────────────────────────────────────────────────────────
    df = pd.read_csv(csv_path, header=None)
    data = df.values.astype(float)          # shape: (rows, cols)

    # ── crop upper half (first ⌈rows/2⌉ rows) ────────────────────────────────
    n_rows, n_cols = data.shape
    half = (n_rows + 1) // 2               # ceiling division → keep upper half
    data_cropped = data[:half, :]

    # ── save cropped CSV ──────────────────────────────────────────────────────
    os.makedirs(out_dir, exist_ok=True)
    base_name   = os.path.splitext(os.path.basename(csv_path))[0]
    csv_out     = os.path.join(out_dir, f"{base_name}.csv")
    pd.DataFrame(data_cropped).to_csv(csv_out, header=False, index=False)
    print(f"    Saved CSV  → {csv_out}")

    # ── plot & save PNG ───────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(
        data_cropped,
        cmap=COLORMAP,
        origin="upper",          # row 0 at the top (upper boundary)
        aspect="auto",
    )
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Temperature (°C)", fontsize=11)
    ax.set_title(
        f"{base_name.replace('_', ' ').title()}  —  {case_label}\n"
        f"(upper half: {half} of {n_rows} rows)",
        fontsize=11,
    )
    ax.set_xlabel("Column index (x-pixel)", fontsize=10)
    ax.set_ylabel("Row index (y-pixel)", fontsize=10)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True, nbins=8))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True, nbins=6))
    plt.tight_layout()

    png_out = os.path.join(out_dir, f"{base_name}.png")
    fig.savefig(png_out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved plot → {png_out}")


def main():
    # Collect all case directories (skip hidden / non-directory entries)
    case_dirs = sorted([
        d for d in glob.glob(os.path.join(ROOT_DIR, "cr_*"))
        if os.path.isdir(d)
    ])

    if not case_dirs:
        print(f"[WARNING] No case folders found under:\n  {ROOT_DIR}")
        return

    print(f"Found {len(case_dirs)} case folder(s).\n")

    for case_dir in case_dirs:
        case_name  = os.path.basename(case_dir)
        case_label = parse_case_name(case_name)
        avg_temp_dir = os.path.join(case_dir, "time_average", "average_temperature")
        out_dir      = os.path.join(avg_temp_dir, OUTPUT_SUBDIR)

        print(f"[Case] {case_name}  ({case_label})")

        if not os.path.isdir(avg_temp_dir):
            print(f"  [SKIP] average_temperature folder not found: {avg_temp_dir}\n")
            continue

        found_any = False
        for fname in CSV_FILES:
            csv_path = os.path.join(avg_temp_dir, fname)
            if not os.path.isfile(csv_path):
                print(f"  [SKIP] {fname} not found in {avg_temp_dir}")
                continue
            found_any = True
            crop_and_save(csv_path, out_dir, case_label)

        if not found_any:
            print("  [SKIP] No CSV files found for this case.")
        print()

    print("Done.")


if __name__ == "__main__":
    main()

Found 15 case folder(s).

[Case] cr_1400_phi_085  (CR = 1400 slpm,  φ = 0.85)
    Saved CSV  → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\window_1_temperature.csv
    Saved plot → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\window_1_temperature.png
    Saved CSV  → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\window_2_temperature.csv
    Saved plot → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\window_2_temperature.png

[Case] cr_1400_phi_09  (CR = 1400 slpm,  φ = 0.90)
    Saved CSV  → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_average\average_temperature\half_domain_temperature\window_1_temperature.csv
    Saved plot → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_average\average_temperature\half_domain_temperature\win